This file is for finding an appropraite architecture
The maximum validation accuracy i could achieve was 98.84% with ResNet34
I will be trying ResNet18, ResNet50, EfficientNet, Convnext, DenseNet, MobileNet, Vision Transformer

In [18]:
import pandas as pd
import regex as re
import torch

In [19]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [20]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return class_id

In [21]:
from pathlib import Path

In [22]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [23]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [24]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [25]:
from torch.utils.data import Dataset
from PIL import Image

In [26]:
class dset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = label_function(Path(image_path))
        if self.transform:
            image = self.transform(image)
        return image, label

In [27]:
image_paths = list(path.rglob("*.png"))

In [28]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = dset(image_paths = image_paths, transform = transform)

In [29]:
from torch.utils.data import random_split

ts = int(0.75*len(dataset))
vs = len(dataset) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset, [ts, vs], generator)

In [30]:
from torch.utils.data import DataLoader

In [31]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

In [33]:
import torchvision
num_classes = 55
device = torch.device("cuda")

In [34]:
import kornia.augmentation as K

In [35]:
import torch.nn as nn

In [36]:
model_1 = torchvision.models.resnet34(weights="DEFAULT")
model_1.fc = nn.Linear(
    model_1.fc.in_features,
    num_classes
)

model_1 = model_1.to(device)

for param in model_1.parameters():
    param.requires_grad = False

for param in model_1.fc.parameters():
    param.requires_grad = True

train_aug_1 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_1 = torch.optim.RMSprop(
    model_1.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_1,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_1.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_1(images)

        optimizer_1.zero_grad()

        outputs = model_1(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_1.step()

        scheduler_1.step()          

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_1.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_1(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 63.70% | Valid Acc: 88.44%
Epoch 2/20 | Train Acc: 88.37% | Valid Acc: 93.06%
Epoch 3/20 | Train Acc: 85.96% | Valid Acc: 87.48%
Epoch 4/20 | Train Acc: 86.77% | Valid Acc: 87.86%
Epoch 5/20 | Train Acc: 88.40% | Valid Acc: 94.32%
Epoch 6/20 | Train Acc: 89.46% | Valid Acc: 90.46%
Epoch 7/20 | Train Acc: 88.98% | Valid Acc: 92.97%
Epoch 8/20 | Train Acc: 89.72% | Valid Acc: 94.89%
Epoch 9/20 | Train Acc: 91.49% | Valid Acc: 89.88%
Epoch 10/20 | Train Acc: 92.16% | Valid Acc: 93.64%
Epoch 11/20 | Train Acc: 90.91% | Valid Acc: 96.05%
Epoch 12/20 | Train Acc: 93.03% | Valid Acc: 94.89%
Epoch 13/20 | Train Acc: 92.68% | Valid Acc: 96.34%
Epoch 14/20 | Train Acc: 94.25% | Valid Acc: 94.22%
Epoch 15/20 | Train Acc: 95.09% | Valid Acc: 95.18%
Epoch 16/20 | Train Acc: 95.86% | Valid Acc: 97.21%
Epoch 17/20 | Train Acc: 97.85% | Valid Acc: 97.88%
Epoch 18/20 | Train Acc: 98.55% | Valid Acc: 98.36%
Epoch 19/20 | Train Acc: 99.20% | Valid Acc: 98.94%
Epoch 20/20 | Train A

In [37]:
model_2 = torchvision.models.resnet18(weights="DEFAULT")
model_2.fc = nn.Linear(
    model_2.fc.in_features,
    num_classes
)

model_2 = model_2.to(device)

for param in model_2.parameters():
    param.requires_grad = False

for param in model_2.fc.parameters():
    param.requires_grad = True

train_aug_2 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_2 = torch.optim.RMSprop(
    model_2.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_2,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_2.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_2(images)

        optimizer_2.zero_grad()

        outputs = model_2(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_2.step()

        scheduler_2.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_2.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_2(images)

            loss = criterion(outputs, labels)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 64.31% | Valid Acc: 88.92%
Epoch 2/20 | Train Acc: 88.47% | Valid Acc: 92.20%
Epoch 3/20 | Train Acc: 86.64% | Valid Acc: 86.99%
Epoch 4/20 | Train Acc: 89.34% | Valid Acc: 92.29%
Epoch 5/20 | Train Acc: 87.09% | Valid Acc: 91.52%
Epoch 6/20 | Train Acc: 88.37% | Valid Acc: 93.26%
Epoch 7/20 | Train Acc: 90.46% | Valid Acc: 88.44%
Epoch 8/20 | Train Acc: 90.62% | Valid Acc: 92.29%
Epoch 9/20 | Train Acc: 89.56% | Valid Acc: 95.18%
Epoch 10/20 | Train Acc: 91.55% | Valid Acc: 91.04%
Epoch 11/20 | Train Acc: 91.29% | Valid Acc: 92.29%
Epoch 12/20 | Train Acc: 94.38% | Valid Acc: 95.66%
Epoch 13/20 | Train Acc: 91.71% | Valid Acc: 94.51%
Epoch 14/20 | Train Acc: 93.93% | Valid Acc: 95.28%
Epoch 15/20 | Train Acc: 95.76% | Valid Acc: 97.01%
Epoch 16/20 | Train Acc: 95.89% | Valid Acc: 96.82%
Epoch 17/20 | Train Acc: 97.75% | Valid Acc: 97.11%
Epoch 18/20 | Train Acc: 98.49% | Valid Acc: 97.98%
Epoch 19/20 | Train Acc: 99.42% | Valid Acc: 98.17%
Epoch 20/20 | Train A

In [38]:
model_3 = torchvision.models.resnet50(weights="DEFAULT")
model_3.fc = nn.Linear(
    model_3.fc.in_features,
    num_classes
)

model_3 = model_3.to(device)

for param in model_3.parameters():
    param.requires_grad = False

for param in model_3.fc.parameters():
    param.requires_grad = True

train_aug_3 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_3 = torch.optim.RMSprop(
    model_3.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_3 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_3,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_3.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_3(images)

        optimizer_3.zero_grad()

        outputs = model_3(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_3.step()

        scheduler_3.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_3.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_3(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 69.96% | Valid Acc: 89.98%
Epoch 2/20 | Train Acc: 93.54% | Valid Acc: 92.29%
Epoch 3/20 | Train Acc: 94.47% | Valid Acc: 96.34%
Epoch 4/20 | Train Acc: 93.70% | Valid Acc: 93.55%
Epoch 5/20 | Train Acc: 93.00% | Valid Acc: 93.74%
Epoch 6/20 | Train Acc: 92.87% | Valid Acc: 96.53%
Epoch 7/20 | Train Acc: 94.22% | Valid Acc: 92.77%
Epoch 8/20 | Train Acc: 93.54% | Valid Acc: 94.12%
Epoch 9/20 | Train Acc: 93.35% | Valid Acc: 92.58%
Epoch 10/20 | Train Acc: 93.61% | Valid Acc: 92.77%
Epoch 11/20 | Train Acc: 95.09% | Valid Acc: 94.22%
Epoch 12/20 | Train Acc: 93.70% | Valid Acc: 93.93%
Epoch 13/20 | Train Acc: 95.47% | Valid Acc: 96.72%
Epoch 14/20 | Train Acc: 95.57% | Valid Acc: 95.28%
Epoch 15/20 | Train Acc: 96.21% | Valid Acc: 95.38%
Epoch 16/20 | Train Acc: 97.75% | Valid Acc: 96.92%
Epoch 17/20 | Train Acc: 98.07% | Valid Acc: 97.69%
Epoch 18/20 | Train Acc: 98.84% | Valid Acc: 97.59%
Epoch 19/20 | Train Acc: 99.58% | Valid Acc: 98.17%
Epoch 20/20 | Train A

In [40]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(next(model_3.parameters()).device)

True
NVIDIA GeForce RTX 4050 Laptop GPU
cuda:0


In [41]:
model_4 = torchvision.models.resnet101(weights="DEFAULT")
model_4.fc = nn.Linear(
    model_4.fc.in_features,
    num_classes
)

model_4 = model_4.to(device)

for param in model_4.parameters():
    param.requires_grad = False

for param in model_4.fc.parameters():
    param.requires_grad = True

train_aug_4 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_4 = torch.optim.RMSprop(
    model_4.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_4 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_4,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_4.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_4(images)

        optimizer_4.zero_grad()

        outputs = model_4(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_4.step()

        scheduler_4.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_4.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_4(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 69.93% | Valid Acc: 90.27%
Epoch 2/20 | Train Acc: 92.29% | Valid Acc: 90.66%
Epoch 3/20 | Train Acc: 94.41% | Valid Acc: 94.12%
Epoch 4/20 | Train Acc: 91.36% | Valid Acc: 94.32%
Epoch 5/20 | Train Acc: 91.04% | Valid Acc: 94.22%
Epoch 6/20 | Train Acc: 91.81% | Valid Acc: 94.32%
Epoch 7/20 | Train Acc: 91.07% | Valid Acc: 93.64%
Epoch 8/20 | Train Acc: 92.07% | Valid Acc: 94.41%
Epoch 9/20 | Train Acc: 92.39% | Valid Acc: 94.51%
Epoch 10/20 | Train Acc: 92.90% | Valid Acc: 94.41%
Epoch 11/20 | Train Acc: 92.00% | Valid Acc: 95.18%
Epoch 12/20 | Train Acc: 93.70% | Valid Acc: 93.93%
Epoch 13/20 | Train Acc: 94.09% | Valid Acc: 94.22%
Epoch 14/20 | Train Acc: 94.44% | Valid Acc: 96.63%
Epoch 15/20 | Train Acc: 96.15% | Valid Acc: 95.86%
Epoch 16/20 | Train Acc: 97.85% | Valid Acc: 97.50%
Epoch 17/20 | Train Acc: 97.75% | Valid Acc: 96.92%
Epoch 18/20 | Train Acc: 98.81% | Valid Acc: 98.55%
Epoch 19/20 | Train Acc: 99.65% | Valid Acc: 98.36%
Epoch 20/20 | Train A

In [42]:
model_5 = torchvision.models.resnet152(weights="DEFAULT")
model_5.fc = nn.Linear(
    model_5.fc.in_features,
    num_classes
)

model_5 = model_5.to(device)

for param in model_5.parameters():
    param.requires_grad = False

for param in model_5.fc.parameters():
    param.requires_grad = True

train_aug_5 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_5 = torch.optim.RMSprop(
    model_5.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_5 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_5,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_5.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_5(images)

        optimizer_5.zero_grad()

        outputs = model_5(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_5.step()

        scheduler_5.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_5.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_5(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Downloading: "https://download.pytorch.org/models/resnet152-f82ba261.pth" to C:\Users\Nilansh Barotia/.cache\torch\hub\checkpoints\resnet152-f82ba261.pth


100%|██████████| 230M/230M [00:11<00:00, 20.8MB/s] 


Epoch 1/20 | Train Acc: 72.95% | Valid Acc: 92.00%
Epoch 2/20 | Train Acc: 94.60% | Valid Acc: 94.12%
Epoch 3/20 | Train Acc: 95.05% | Valid Acc: 95.28%
Epoch 4/20 | Train Acc: 92.03% | Valid Acc: 92.20%
Epoch 5/20 | Train Acc: 93.54% | Valid Acc: 92.39%
Epoch 6/20 | Train Acc: 92.48% | Valid Acc: 92.49%
Epoch 7/20 | Train Acc: 92.93% | Valid Acc: 92.58%
Epoch 8/20 | Train Acc: 92.93% | Valid Acc: 93.45%
Epoch 9/20 | Train Acc: 92.93% | Valid Acc: 93.74%
Epoch 10/20 | Train Acc: 93.29% | Valid Acc: 95.95%
Epoch 11/20 | Train Acc: 94.38% | Valid Acc: 94.12%
Epoch 12/20 | Train Acc: 94.47% | Valid Acc: 94.51%
Epoch 13/20 | Train Acc: 94.57% | Valid Acc: 95.76%
Epoch 14/20 | Train Acc: 96.27% | Valid Acc: 96.24%
Epoch 15/20 | Train Acc: 97.37% | Valid Acc: 96.82%
Epoch 16/20 | Train Acc: 96.79% | Valid Acc: 97.11%
Epoch 17/20 | Train Acc: 98.39% | Valid Acc: 97.21%
Epoch 18/20 | Train Acc: 99.23% | Valid Acc: 98.07%
Epoch 19/20 | Train Acc: 99.39% | Valid Acc: 98.36%
Epoch 20/20 | Train A

In [43]:
model_6 = torchvision.models.efficientnet_b0(weights="DEFAULT")
model_6.classifier[1] = nn.Linear(
    model_6.classifier[1].in_features,
    num_classes
)

model_6 = model_6.to(device)

for param in model_6.parameters():
    param.requires_grad = False

for param in model_6.classifier.parameters():
    param.requires_grad = True

train_aug_6 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_6 = torch.optim.RMSprop(
    model_6.classifier.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_6 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_6,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_6.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_6(images)

        optimizer_6.zero_grad()

        outputs = model_6(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_6.step()

        scheduler_6.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_6.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_6(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Nilansh Barotia/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:01<00:00, 20.0MB/s]


Epoch 1/20 | Train Acc: 68.55% | Valid Acc: 90.37%
Epoch 2/20 | Train Acc: 91.26% | Valid Acc: 94.41%
Epoch 3/20 | Train Acc: 90.91% | Valid Acc: 93.74%
Epoch 4/20 | Train Acc: 90.78% | Valid Acc: 93.45%
Epoch 5/20 | Train Acc: 90.20% | Valid Acc: 93.16%
Epoch 6/20 | Train Acc: 87.66% | Valid Acc: 95.95%
Epoch 7/20 | Train Acc: 89.50% | Valid Acc: 95.47%
Epoch 8/20 | Train Acc: 90.20% | Valid Acc: 95.66%
Epoch 9/20 | Train Acc: 90.33% | Valid Acc: 94.89%
Epoch 10/20 | Train Acc: 90.52% | Valid Acc: 95.76%
Epoch 11/20 | Train Acc: 90.30% | Valid Acc: 94.99%
Epoch 12/20 | Train Acc: 90.62% | Valid Acc: 95.86%
Epoch 13/20 | Train Acc: 91.07% | Valid Acc: 96.92%
Epoch 14/20 | Train Acc: 92.26% | Valid Acc: 95.76%
Epoch 15/20 | Train Acc: 94.22% | Valid Acc: 96.53%
Epoch 16/20 | Train Acc: 94.47% | Valid Acc: 96.82%
Epoch 17/20 | Train Acc: 95.60% | Valid Acc: 97.69%
Epoch 18/20 | Train Acc: 96.18% | Valid Acc: 97.59%
Epoch 19/20 | Train Acc: 97.78% | Valid Acc: 98.07%
Epoch 20/20 | Train A

In [44]:
model_7 = torchvision.models.efficientnet_v2_s(weights="DEFAULT")
model_7.classifier[1] = nn.Linear(
    model_7.classifier[1].in_features,
    num_classes
)

model_7 = model_7.to(device)

for param in model_7.parameters():
    param.requires_grad = False

for param in model_7.classifier.parameters():
    param.requires_grad = True

train_aug_7 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_7 = torch.optim.RMSprop(
    model_7.classifier.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_7 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_7,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_7.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_7(images)

        optimizer_7.zero_grad()

        outputs = model_7(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_7.step()

        scheduler_7.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_7.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_7(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to C:\Users\Nilansh Barotia/.cache\torch\hub\checkpoints\efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:03<00:00, 22.4MB/s]


Epoch 1/20 | Train Acc: 66.95% | Valid Acc: 88.92%
Epoch 2/20 | Train Acc: 88.21% | Valid Acc: 91.43%
Epoch 3/20 | Train Acc: 87.38% | Valid Acc: 91.33%
Epoch 4/20 | Train Acc: 86.28% | Valid Acc: 93.26%
Epoch 5/20 | Train Acc: 85.93% | Valid Acc: 90.37%
Epoch 6/20 | Train Acc: 87.34% | Valid Acc: 92.29%
Epoch 7/20 | Train Acc: 85.26% | Valid Acc: 91.52%
Epoch 8/20 | Train Acc: 87.18% | Valid Acc: 93.06%
Epoch 9/20 | Train Acc: 88.02% | Valid Acc: 92.00%
Epoch 10/20 | Train Acc: 87.38% | Valid Acc: 93.74%
Epoch 11/20 | Train Acc: 86.60% | Valid Acc: 93.26%
Epoch 12/20 | Train Acc: 89.05% | Valid Acc: 92.58%
Epoch 13/20 | Train Acc: 88.11% | Valid Acc: 92.87%
Epoch 14/20 | Train Acc: 89.21% | Valid Acc: 94.61%
Epoch 15/20 | Train Acc: 90.88% | Valid Acc: 94.41%
Epoch 16/20 | Train Acc: 92.55% | Valid Acc: 96.15%
Epoch 17/20 | Train Acc: 93.48% | Valid Acc: 96.53%
Epoch 18/20 | Train Acc: 94.80% | Valid Acc: 96.92%
Epoch 19/20 | Train Acc: 96.24% | Valid Acc: 97.40%
Epoch 20/20 | Train A

In [45]:
model_8 = torchvision.models.convnext_tiny(weights="DEFAULT")
model_8.classifier[2] = nn.Linear(
    model_8.classifier[2].in_features,
    num_classes
)

model_8 = model_8.to(device)

for param in model_8.parameters():
    param.requires_grad = False

for param in model_8.classifier.parameters():
    param.requires_grad = True

train_aug_8 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_8 = torch.optim.RMSprop(
    model_8.classifier.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_8 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_8,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_8.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_8(images)

        optimizer_8.zero_grad()

        outputs = model_8(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_8.step()

        scheduler_8.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_8.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_8(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to C:\Users\Nilansh Barotia/.cache\torch\hub\checkpoints\convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:06<00:00, 18.8MB/s] 


Epoch 1/20 | Train Acc: 66.05% | Valid Acc: 86.32%
Epoch 2/20 | Train Acc: 89.11% | Valid Acc: 93.83%
Epoch 3/20 | Train Acc: 92.74% | Valid Acc: 93.16%
Epoch 4/20 | Train Acc: 92.52% | Valid Acc: 94.03%
Epoch 5/20 | Train Acc: 92.87% | Valid Acc: 92.20%
Epoch 6/20 | Train Acc: 92.13% | Valid Acc: 94.61%
Epoch 7/20 | Train Acc: 93.22% | Valid Acc: 93.16%
Epoch 8/20 | Train Acc: 93.00% | Valid Acc: 92.77%
Epoch 9/20 | Train Acc: 93.96% | Valid Acc: 94.32%
Epoch 10/20 | Train Acc: 94.41% | Valid Acc: 95.47%
Epoch 11/20 | Train Acc: 94.41% | Valid Acc: 94.89%
Epoch 12/20 | Train Acc: 95.47% | Valid Acc: 95.09%
Epoch 13/20 | Train Acc: 94.70% | Valid Acc: 96.44%
Epoch 14/20 | Train Acc: 95.95% | Valid Acc: 95.86%
Epoch 15/20 | Train Acc: 96.47% | Valid Acc: 97.01%
Epoch 16/20 | Train Acc: 97.17% | Valid Acc: 97.01%
Epoch 17/20 | Train Acc: 97.65% | Valid Acc: 96.34%
Epoch 18/20 | Train Acc: 98.07% | Valid Acc: 97.30%
Epoch 19/20 | Train Acc: 98.43% | Valid Acc: 97.01%
Epoch 20/20 | Train A

In [46]:
model_9 = torchvision.models.densenet121(weights="DEFAULT")
model_9.classifier = nn.Linear(
    model_9.classifier.in_features,
    num_classes
)

model_9 = model_9.to(device)

for param in model_9.parameters():
    param.requires_grad = False

for param in model_9.classifier.parameters():
    param.requires_grad = True

train_aug_9 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_9 = torch.optim.RMSprop(
    model_9.classifier.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_9 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_9,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_9.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_9(images)

        optimizer_9.zero_grad()

        outputs = model_9(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_9.step()

        scheduler_9.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_9.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_9(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\Nilansh Barotia/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:02<00:00, 11.0MB/s]


Epoch 1/20 | Train Acc: 66.53% | Valid Acc: 90.94%
Epoch 2/20 | Train Acc: 88.21% | Valid Acc: 93.35%
Epoch 3/20 | Train Acc: 89.75% | Valid Acc: 88.82%
Epoch 4/20 | Train Acc: 88.66% | Valid Acc: 92.20%
Epoch 5/20 | Train Acc: 88.76% | Valid Acc: 92.97%
Epoch 6/20 | Train Acc: 91.33% | Valid Acc: 92.39%
Epoch 7/20 | Train Acc: 90.56% | Valid Acc: 93.93%
Epoch 8/20 | Train Acc: 89.82% | Valid Acc: 94.80%
Epoch 9/20 | Train Acc: 92.45% | Valid Acc: 91.33%
Epoch 10/20 | Train Acc: 92.77% | Valid Acc: 95.38%
Epoch 11/20 | Train Acc: 92.03% | Valid Acc: 94.80%
Epoch 12/20 | Train Acc: 92.23% | Valid Acc: 94.51%
Epoch 13/20 | Train Acc: 94.96% | Valid Acc: 95.66%
Epoch 14/20 | Train Acc: 94.64% | Valid Acc: 96.72%
Epoch 15/20 | Train Acc: 95.41% | Valid Acc: 97.69%
Epoch 16/20 | Train Acc: 96.92% | Valid Acc: 97.50%
Epoch 17/20 | Train Acc: 98.30% | Valid Acc: 98.46%
Epoch 18/20 | Train Acc: 98.65% | Valid Acc: 98.75%
Epoch 19/20 | Train Acc: 99.42% | Valid Acc: 98.55%
Epoch 20/20 | Train A

In [47]:
model_10 = torchvision.models.mobilenet_v3_large(weights="DEFAULT")
model_10.classifier[3] = nn.Linear(
    model_10.classifier[3].in_features,
    num_classes
)

model_10 = model_10.to(device)

for param in model_10.parameters():
    param.requires_grad = False

for param in model_10.classifier.parameters():
    param.requires_grad = True

train_aug_10 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_10 = torch.optim.RMSprop(
    model_10.classifier.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_10 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_10,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_10.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_10(images)

        optimizer_10.zero_grad()

        outputs = model_10(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_10.step()

        scheduler_10.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_10.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_10(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to C:\Users\Nilansh Barotia/.cache\torch\hub\checkpoints\mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 24.3MB/s]


Epoch 1/20 | Train Acc: 70.90% | Valid Acc: 86.80%
Epoch 2/20 | Train Acc: 76.74% | Valid Acc: 83.04%
Epoch 3/20 | Train Acc: 71.70% | Valid Acc: 74.37%
Epoch 4/20 | Train Acc: 73.79% | Valid Acc: 76.59%
Epoch 5/20 | Train Acc: 76.33% | Valid Acc: 86.42%
Epoch 6/20 | Train Acc: 80.69% | Valid Acc: 85.16%
Epoch 7/20 | Train Acc: 80.31% | Valid Acc: 87.57%
Epoch 8/20 | Train Acc: 81.75% | Valid Acc: 87.86%
Epoch 9/20 | Train Acc: 84.84% | Valid Acc: 89.50%
Epoch 10/20 | Train Acc: 84.03% | Valid Acc: 88.34%
Epoch 11/20 | Train Acc: 85.48% | Valid Acc: 92.39%
Epoch 12/20 | Train Acc: 87.95% | Valid Acc: 92.20%
Epoch 13/20 | Train Acc: 87.54% | Valid Acc: 93.93%
Epoch 14/20 | Train Acc: 87.12% | Valid Acc: 93.45%
Epoch 15/20 | Train Acc: 91.62% | Valid Acc: 94.41%
Epoch 16/20 | Train Acc: 91.84% | Valid Acc: 95.18%
Epoch 17/20 | Train Acc: 94.35% | Valid Acc: 96.72%
Epoch 18/20 | Train Acc: 96.63% | Valid Acc: 97.69%
Epoch 19/20 | Train Acc: 97.43% | Valid Acc: 97.59%
Epoch 20/20 | Train A

In [48]:
model_11 = torchvision.models.vit_b_16(weights="DEFAULT")
model_11.heads.head = nn.Linear(
    model_11.heads.head.in_features,
    num_classes
)

model_11 = model_11.to(device)

for param in model_11.parameters():
    param.requires_grad = False

for param in model_11.heads.head.parameters():
    param.requires_grad = True

train_aug_11 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

optimizer_11 = torch.optim.RMSprop(
    model_11.heads.head.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_11 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_11,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_11.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_11(images)

        optimizer_11.zero_grad()

        outputs = model_11(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_11.step()

        scheduler_11.step()          

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total


    model_11.eval()

    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_11(images)

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\Nilansh Barotia/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:15<00:00, 22.7MB/s] 


Epoch 1/20 | Train Acc: 66.27% | Valid Acc: 86.13%
Epoch 2/20 | Train Acc: 90.94% | Valid Acc: 91.52%
Epoch 3/20 | Train Acc: 90.91% | Valid Acc: 91.33%
Epoch 4/20 | Train Acc: 92.00% | Valid Acc: 86.22%
Epoch 5/20 | Train Acc: 90.14% | Valid Acc: 86.32%
Epoch 6/20 | Train Acc: 90.52% | Valid Acc: 88.54%
Epoch 7/20 | Train Acc: 92.00% | Valid Acc: 87.38%
Epoch 8/20 | Train Acc: 92.74% | Valid Acc: 86.61%
Epoch 9/20 | Train Acc: 91.74% | Valid Acc: 91.71%
Epoch 10/20 | Train Acc: 92.80% | Valid Acc: 89.60%
Epoch 11/20 | Train Acc: 93.74% | Valid Acc: 86.03%
Epoch 12/20 | Train Acc: 93.45% | Valid Acc: 90.08%
Epoch 13/20 | Train Acc: 93.90% | Valid Acc: 93.93%
Epoch 14/20 | Train Acc: 95.18% | Valid Acc: 94.32%
Epoch 15/20 | Train Acc: 96.59% | Valid Acc: 94.12%
Epoch 16/20 | Train Acc: 96.95% | Valid Acc: 96.15%
Epoch 17/20 | Train Acc: 98.81% | Valid Acc: 97.59%
Epoch 18/20 | Train Acc: 99.52% | Valid Acc: 97.30%
Epoch 19/20 | Train Acc: 99.90% | Valid Acc: 97.78%
Epoch 20/20 | Train A

So Resnet34 is perfect for our case as we are only training head and out dataset is small 